In [12]:
# basic imports
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import datetime, os
from cartopy import crs as ccrs
from cartopy import feature as cfeature
import sys
import re
from IPython.display import clear_output
import datetime
import textwrap
import csv

#import special functions
from utils.infilling import infill_missing_data, calculate_statistical_significance

pd.options.display.max_columns = 50
print("Last updated on {}".format(datetime.datetime.now().ctime()))

Last updated on Fri Dec  5 14:57:15 2025


In [13]:
# setup directories ---
current_directory = os.getcwd()
Figures = os.path.join(current_directory, 'figures')
Files = os.path.join(current_directory, 'output_txt_files')
CSV = os.path.join(current_directory, 'csv_files')
PKL = os.path.join(current_directory, 'PKL')
for folder in [Figures, Files, CSV]:
    if not os.path.exists(folder):
        os.makedirs(folder)

In [14]:
# load cleaned, tier1 dataset
print("Loading and preparing Tier 1 data...")
try:
    df = pd.read_pickle(os.path.join(PKL, 'Tier1.pkl'))
    print("Tier 1 loaded.")
except FileNotFoundError:
    print("Error: Tier1.pkl' not found.")

# Initial data sorting
separator = '###'
df['Landmark'] = df['Landmark'].str.replace(r'\s+', separator, regex=True)
df = df.sort_values(by=['LogBook ID', 'Entry Date Time', 'Landmark'])
df['Landmark'] = df['Landmark'].str.replace(separator, ' ')

# Initialize 'Infilled' column
df['Infilled'] = False

Loading and preparing Tier 1 data...
Tier 1 loaded.


In [15]:
# Define initial usable data
df['usable'] = (df["Latitude_decimal"].notna() & df["Longitude_decimal"].notna() & df["BF Value"].notna())
df_u1 = df[df['usable']==True].copy()
print(f"   - Initial usable entries (Tier 1): {len(df_u1)}")

   - Initial usable entries (Tier 1): 69141


In [16]:
# infill and print stats

# set tier1 as baseline for stat testing
baseline_label  = "Tier 1 (baseline)"
baseline_series = pd.to_numeric(df_u1['BF Value'], errors='coerce').dropna()

# define different levels of infilling 
infilling_tiers = [
    {'tier': 2, 'days_missing': 1, 'max_dist_km': 350, 'max_dist_deg': 2.01},
    {'tier': 3, 'days_missing': 2, 'max_dist_km': 400, 'max_dist_deg': 3.01},
    {'tier': 4, 'days_missing': 3, 'max_dist_km': 450, 'max_dist_deg': 4.01},
    {'tier': 4, 'days_missing': 4, 'max_dist_km': 450, 'max_dist_deg': 4.01},
    {'tier': 4, 'days_missing': 5, 'max_dist_km': 450, 'max_dist_deg': 4.01},
]

In [17]:
print("Starting sequential infilling process...")
current_df = df.copy()

# keep snapshots for later saving 
tier_dataframes = {1: df_u1}   
tier_full_snapshots = {}        
tier4_step_usable_counts = {}  

KM_PER_DEG_LAT = 111.32  # mean km per degree latitude

for idx, tier_info in enumerate(infilling_tiers):
    tier = tier_info['tier']
    days = tier_info['days_missing']
    km   = tier_info['max_dist_km']   # already in km (for latlon)
    deg  = tier_info['max_dist_deg']  # degrees (for single coord thresholds in your old code)

    print(f"\n--- Processing Tier {tier} (gaps of {days} days) ---")

    # 1) paired lat+lon (threshold already in km)
    current_df = infill_missing_data(current_df, ['Latitude_decimal', 'Longitude_decimal'], days, km)

    # 2) lat-only: convert deg -> km
    lat_km = deg * KM_PER_DEG_LAT
    current_df = infill_missing_data(current_df, ['Latitude_decimal'], days, lat_km)

    # 3) lon-only: also pass km; use a conservative conversion (equator upper bound)
    lon_km = deg * KM_PER_DEG_LAT
    current_df = infill_missing_data(current_df, ['Longitude_decimal'], days, lon_km)

    # for Tier 4 immediately compute and print per-step stats
    if tier == 4:
        usable_mask = (
            current_df["Latitude_decimal"].notna() &
            current_df["Longitude_decimal"].notna() &
            current_df["BF Value"].notna()
        )
        tier4_step_usable_counts[days] = int(usable_mask.sum())

        # per-step statistical tests vs baseline
        print("\n=== Tier 4 usable-entry totals by days of allowed gap ===")
        sample_series = pd.to_numeric(current_df.loc[usable_mask, 'BF Value'], errors='coerce').dropna()
        calculate_statistical_significance(
            baseline_series,
            sample_series,
            "Tier 1 (baseline)",
            f"Tier 4, {days} days"
        )
        print(f"  After allowing gaps up to {days} days: {tier4_step_usable_counts[days]:,} usable entries")

    # is this the last entry for this tier number?
    is_last_for_tier = (
        idx == len(infilling_tiers) - 1
        or infilling_tiers[idx + 1]['tier'] != tier
    )

    if is_last_for_tier:
        print(f"--- Finalizing Tier {tier} ---")
        usable_col = f'Tier{tier}_usable'
        current_df[usable_col] = (
            current_df["Latitude_decimal"].notna() &
            current_df["Longitude_decimal"].notna() &
            current_df["BF Value"].notna()
        )
        df_usable_current = current_df[current_df[usable_col]].copy()
        tier_dataframes[tier] = df_usable_current               # final usable per tier
        tier_full_snapshots[tier] = current_df.copy()           # full dataframe snapshot for later saving

        # final tier-level tests vs baseline (Tier 2, Tier 3, and final Tier 4)
        sample_series = pd.to_numeric(df_usable_current['BF Value'], errors='coerce').dropna()
        calculate_statistical_significance(
            baseline_series,
            sample_series,
            "Tier 1 (baseline)",
            f"Tier {tier}"
        )
        print(f"  - Tier {tier} complete. Usable entries: {len(df_usable_current)}")

Starting sequential infilling process...

--- Processing Tier 2 (gaps of 1 days) ---
  - Infilling 1-day gaps for columns: ['Latitude_decimal', 'Longitude_decimal']...
  - Infilling 1-day gaps for columns: ['Latitude_decimal']...
  - Infilling 1-day gaps for columns: ['Longitude_decimal']...
--- Finalizing Tier 2 ---
  - Running statistical tests for 'Tier 1 (baseline)' vs. 'Tier 2'...
    - Mann Whitney U Test: MannwhitneyuResult(statistic=2542583618.5, pvalue=0.4357274309059799)
    - Kruskal-Wallis Test: KruskalResult(statistic=0.6075086504285325, pvalue=0.4357273922344108)
  - Tier 2 complete. Usable entries: 73376

--- Processing Tier 3 (gaps of 2 days) ---
  - Infilling 2-day gaps for columns: ['Latitude_decimal', 'Longitude_decimal']...
  - Infilling 2-day gaps for columns: ['Latitude_decimal']...
  - Infilling 2-day gaps for columns: ['Longitude_decimal']...
--- Finalizing Tier 3 ---
  - Running statistical tests for 'Tier 1 (baseline)' vs. 'Tier 3'...
    - Mann Whitney U Test

In [18]:
#added entries
print(73376 - 69141)  # Tier 2 vs baseline
print(75179 - 73376)  # Tier 3 vs Tier 2
print(76110 - 75179)  # Tier 4 (3d) vs Tier 3
print(76705 - 76110)  # Tier 4 (4d) vs Tier 4 (3d)
print(77071 - 76705)  # Tier 4 (5d) vs Tier 4 (4d)

#percent increase
print(73376/69141)
print(75179/69141)
print(76110/69141)
print(76705/69141)
print(77071/69141)

4235
1803
931
595
366
1.061251645188817
1.0873287918890384
1.1007940295917038
1.1093996326347608
1.1146931632461203


In [19]:
# drop unnecessary columns
from utils.infilling import drop_cols_if_present
columns_to_drop = ['Direction', 'wind force', 'Local Time', 'Depth_og']

# save modified dataframes
for tier in sorted(tier_full_snapshots):
    tier_df = tier_full_snapshots[tier]
    tier_df = drop_cols_if_present(tier_df, columns_to_drop)

    fname_base = f"Tier{tier}"
    print(f"Saving {fname_base}...")
    tier_df.to_pickle(os.path.join(PKL, f"{fname_base}.pkl"))
    tier_df.to_csv(os.path.join(CSV, f"{fname_base}.csv"), index=False)

Saving Tier2...
Saving Tier3...
Saving Tier4...


In [2]:
#upload all files to drive using bash utility script
!./utils/sync_data.sh